# HTR-VT (CNN + transformer + CTC, from scratch), greedy

Our implementation after HTR-VT (Li et al., Pattern Recognition 2025): residual CNN front-end, span masking, 6-layer transformer encoder (dim 384), linear CTC head, trained from scratch, 300 epochs fixed (dev curve logged to check the budget). Augmentation study, wave 1. Data `cyttic/hhd-optionb-split`: train = Option-B train minus the dev writers (20%, seed 42); online augmentation `COND` from `augment.py`; dev scored every few epochs (learning curve only); the HHD Option-B test scored once after the last epoch, **greedy CTC decoding, no language model** (CER, CER without punctuation, WER).

Parameters (papermill): `COND`, `SEED`, `REPO_DIR`, `PUSH`, `SMOKE`. Output: public HF repo `cyttic/hmt-<model>-<COND>-s<SEED>` (results.json, train_log.tsv, test_predictions.tsv, weights).

In [ ]:
COND = "none"            # "none" or <family><level>, family in geom / elastic / stroke / photo, level 1..4 (augment.py)
SEED = 42
REPO_DIR = "~/htr-metrics-transfer"
PUSH, SMOKE = True, False

In [ ]:
!pip install -q "transformers==5.15.0" datasets jiwer opencv-python-headless safetensors

In [ ]:
import os, math
MODEL = "htrvt"
CONFIG = {"row_h": 64, "max_w": 2048, "batch": 32, "lr": 5e-4, "warmup": 0.05, "wd": 0.05, "epochs": 300, "eval_every": 10,
          "cnn_width": 256, "dim": 384, "depth": 6, "heads": 6, "mlp_ratio": 4, "dropout": 0.1, "mask_ratio": 0.4, "mask_span": 4,
          "schedule": "linear warm-up + cosine decay to 1% of the peak"}
ROW_H, MAX_W, BATCH = CONFIG["row_h"], CONFIG["max_w"], CONFIG["batch"]
LR, WARMUP, WD, EPOCHS, EVAL_EVERY = CONFIG["lr"], CONFIG["warmup"], CONFIG["wd"], CONFIG["epochs"], CONFIG["eval_every"]
NUM_WORKERS, EVAL_BS = min(8, os.cpu_count()), 32
def schedule(step, total, warm):
    if step < warm: return (step + 1) / warm
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))

In [ ]:
import os, sys, io, re, json, math, time, random, unicodedata
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, jiwer
from PIL import Image
from datasets import load_dataset
from huggingface_hub import hf_hub_download, HfApi, get_token, whoami
assert get_token() is not None, "run `hf auth login` on the server first"
print("HF user:", whoami()["name"])
sys.path.insert(0, os.path.expanduser(REPO_DIR))
import augment as A   # the SAME file the metric computation uses
FAMILY, LEVEL = A.parse_condition(COND)
device = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.bfloat16 if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| amp", AMP_DTYPE, "| condition", COND, (FAMILY, LEVEL), "| seed", SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT = os.path.abspath(f"run_{MODEL}_{COND}_s{SEED}"); os.makedirs(OUT, exist_ok=True)
OUT_REPO = f"cyttic/hmt-{MODEL}-{COND}-s{SEED}"   # public
print("OUT", OUT, "->", OUT_REPO)

In [ ]:
STRIDE = 4   # the CNN reduces width 4x -> one CTC step per 4 px
def fmt(g):
    """64 px gray line -> mirror (RTL -> LTR) -> width capped at MAX_W -> (1,64,W) in [-1,1] (ink dark) + CTC length"""
    g = g[:, ::-1]
    if g.shape[1] > MAX_W: g = A.cv2.resize(np.ascontiguousarray(g), (MAX_W, ROW_H), interpolation=A.cv2.INTER_AREA)
    t = torch.from_numpy(np.ascontiguousarray(g)).float().div(255).sub(0.5).div(0.5).unsqueeze(0)
    return t, max(1, t.shape[-1] // STRIDE)
def collate(b):
    W = max(x[0].shape[-1] for x in b); W = int(math.ceil(W / STRIDE) * STRIDE)
    px = torch.ones(len(b), 1, ROW_H, W)   # pad with white (+1)
    for i, x in enumerate(b): px[i, :, :, :x[0].shape[-1]] = x[0]
    return px, torch.tensor([x[1] for x in b]), [x[2] for x in b], [x[3] for x in b]

In [ ]:
CHARS = json.load(open(hf_hub_download("cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct", "charset.json")))["chars"]
assert CHARS[0] == ""   # class 0 = CTC blank; the same 198 classes as exp32-39 / Option B
C2I = {c: i for i, c in enumerate(CHARS) if i}; N_CLASSES = len(CHARS)
def encode(text): return [C2I[c] for c in text if c in C2I]
def greedy(lp, n):
    best = lp[:n].argmax(-1).tolist(); out, prev = [], 0
    for k in best:
        if k != prev and k != 0: out.append(CHARS[k])
        prev = k
    return "".join(out)

hhd = load_dataset("cyttic/hhd-optionb-split")
train_full, test = hhd["train"], hhd["test"]
tw = list(train_full["writer"]); wr = sorted(set(tw)); random.Random(42).shuffle(wr); dev_w = set(wr[:int(len(wr) * 0.20)])   # FIXED dev
dev = train_full.select([i for i, w in enumerate(tw) if w in dev_w])
train = train_full.select([i for i, w in enumerate(tw) if w not in dev_w])
assert not set(train["writer"]) & set(test["writer"]) and not set(train["paragraph"]) & set(test["paragraph"])
print(f"train {len(train)} lines / {len(set(train['writer']))} writers | dev {len(dev)} / {len(dev_w)} | test {len(test)} / {len(set(test['writer']))}")
if SMOKE:
    train, dev, test = train.select(range(64)), dev.select(range(32)), test.select(range(32)); EPOCHS, EVAL_EVERY = 2, 1

class LineDS(torch.utils.data.Dataset):
    """HF split -> (model input, CTC length, label ids, text). Augmentation (train only) on the 64 px grayscale line, then fmt()."""
    def __init__(self, split, aug): self.s, self.aug, self.rng = split, aug, np.random.default_rng(SEED)
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        ex = self.s[i]; g = A.to_line(ex["image"], ROW_H)
        if self.aug: g = A.augment(g, FAMILY, LEVEL, self.rng)
        x, n = fmt(g)
        return x, n, encode(ex["text"]), ex["text"]

def loader(split, train_mode, epoch=0):
    ds = LineDS(split, aug=train_mode and LEVEL > 0)
    def init(wid):   # a distinct, reproducible augmentation stream per (seed, epoch, worker)
        torch.utils.data.get_worker_info().dataset.rng = np.random.default_rng([SEED, epoch, wid])
    return torch.utils.data.DataLoader(ds, batch_size=BATCH if train_mode else EVAL_BS, shuffle=train_mode, num_workers=NUM_WORKERS,
                                       collate_fn=collate, worker_init_fn=init, drop_last=False)

def scores(R, H):
    """CER on the raw text, CER without punctuation (Unicode P* removed from both sides), WER. Empty references skipped."""
    nop = lambda t: re.sub(r" +", " ", "".join(c for c in t if not unicodedata.category(c).startswith("P"))).strip()
    k = [i for i, r in enumerate(R) if r.strip()]; k2 = [i for i in k if nop(R[i])]
    return {"cer": jiwer.cer([R[i] for i in k], [H[i] for i in k]), "wer": jiwer.wer([R[i] for i in k], [H[i] for i in k]),
            "cer_nopunct": jiwer.cer([nop(R[i]) for i in k2], [nop(H[i]) for i in k2])}

@torch.no_grad()
def predict(model, split):
    model.eval(); H, R = [], []
    for x, n, _, texts in loader(split, False):
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
            lp = model(x.to(device), n.to(device))
        lp = lp.float().cpu().numpy(); H += [greedy(lp[i], int(n[i])) for i in range(len(texts))]; R += texts
    return R, H

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, cin, cout, pool):
        super().__init__()
        self.c1, self.b1 = nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout)
        self.c2, self.b2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout)
        self.sc = nn.Identity() if cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout))
        self.pool = nn.MaxPool2d(pool) if pool != (1, 1) else nn.Identity()
    def forward(self, x):
        y = self.b2(self.c2(F.relu(self.b1(self.c1(x)))))
        return self.pool(F.relu(y + self.sc(x)))

class HTRVT(nn.Module):
    """Our implementation after HTR-VT (Li et al., Pattern Recognition 2025): a residual CNN turns the 64 px line into a feature
    sequence (height 64 -> 4, folded into channels; width / 4), span masking on that sequence during training, sinusoidal positions,
    a pre-norm transformer encoder, a linear CTC head. Trained from scratch."""
    def __init__(self, n_classes, cnn_width, dim, depth, heads, mlp_ratio, dropout, mask_ratio, mask_span, **_):
        super().__init__()
        w1, w2, w3 = cnn_width // 4, cnn_width // 2, cnn_width
        self.cnn = nn.Sequential(nn.Conv2d(1, w1, 3, padding=1, bias=False), nn.BatchNorm2d(w1), nn.ReLU(),
                                 ResBlock(w1, w1, (2, 2)), ResBlock(w1, w2, (2, 2)), ResBlock(w2, w2, (2, 1)),
                                 ResBlock(w2, w3, (2, 1)), ResBlock(w3, w3, (1, 1)))
        self.proj = nn.Linear(w3 * 4, dim); self.mask_ratio, self.mask_span = mask_ratio, mask_span
        self.drop = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(dim, heads, int(dim * mlp_ratio), dropout, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm, self.head = nn.LayerNorm(dim), nn.Linear(dim, n_classes)
        pe = torch.zeros(4096, dim); pos = torch.arange(4096).float()[:, None]
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div); self.register_buffer("pe", pe, persistent=False)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.trunc_normal_(m.weight, std=0.02); nn.init.zeros_(m.bias) if m.bias is not None else None
    def span_mask(self, x):
        B, T, _ = x.shape; n = max(1, int(T * self.mask_ratio / self.mask_span))
        st = torch.randint(0, max(1, T - self.mask_span), (B, n), device=x.device); m = torch.zeros(B, T, dtype=torch.bool, device=x.device)
        for o in range(self.mask_span): m.scatter_(1, (st + o).clamp(max=T - 1), True)
        return x.masked_fill(m[..., None], 0.0)
    def forward(self, px, lens=None):   # lens = CTC steps per line; padded steps are masked out of attention
        f = self.cnn(px); B, Cc, Hh, W = f.shape
        x = self.proj(f.permute(0, 3, 1, 2).reshape(B, W, Cc * Hh))
        if self.training and self.mask_ratio > 0: x = self.span_mask(x)
        pad = None if lens is None else torch.arange(W, device=x.device)[None, :] >= lens[:, None]
        x = self.enc(self.drop(x + self.pe[:W]), src_key_padding_mask=pad)
        return F.log_softmax(self.head(self.norm(x)).float(), dim=-1)

torch.manual_seed(SEED)
model = HTRVT(N_CLASSES, **CONFIG).to(device)
PARAM_GROUPS = [{"params": list(model.parameters()), "lr": LR}]
print(f"HTR-VT (ours) {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, trained from scratch")

In [ ]:
def train_model(model, param_groups):
    opt = torch.optim.AdamW(param_groups, weight_decay=WD); base_lr = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda" and AMP_DTYPE == torch.float16)
    spe = math.ceil(len(train) / BATCH); total = EPOCHS * spe; warm = max(1, int(WARMUP * total)); step = 0; t0 = time.time(); curve = []
    log = open(os.path.join(OUT, "train_log.tsv"), "w"); log.write("epoch\tstep\ttrain_loss\tdev_cer\tdev_wer\tsec\n")
    print(f"{len(train)} train lines | {spe} steps/epoch | {EPOCHS} epochs = {total} steps | augmentation {COND}")
    for ep in range(1, EPOCHS + 1):
        model.train(); losses = []
        for x, n, labels, _ in loader(train, True, ep):
            keep = [i for i, l in enumerate(labels) if 0 < len(l) <= int(n[i])]
            if not keep: continue
            f = schedule(step, total, warm)
            for g, b in zip(opt.param_groups, base_lr): g["lr"] = b * f
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
                lp = model(x[keep].to(device), n[keep].to(device))
            lab = [labels[i] for i in keep]
            loss = F.ctc_loss(lp.float().transpose(0, 1), torch.tensor(sum(lab, [])), n[keep], torch.tensor([len(l) for l in lab]),
                              blank=0, zero_infinity=True)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            losses.append(float(loss)); step += 1
        tl = float(np.mean(losses)); row = {"epoch": ep, "step": step, "train_loss": tl}
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            R, H = predict(model, dev); row.update({f"dev_{k}": v for k, v in scores(R, H).items()})
        curve.append(row)
        log.write(f"{ep}\t{step}\t{tl:.4f}\t{row.get('dev_cer', '')}\t{row.get('dev_wer', '')}\t{time.time()-t0:.0f}\n"); log.flush()
        print(f"[epoch {ep}/{EPOCHS}] loss {tl:.4f}" + (f" | dev CER {100*row['dev_cer']:.2f}" if "dev_cer" in row else "") +
              f" | {time.time()-t0:.0f}s", flush=True)
    return curve, time.time() - t0

In [ ]:
curve, sec = train_model(model, PARAM_GROUPS)
R, H = predict(model, test); T = scores(R, H)
print("HHD Option-B TEST (greedy, last epoch):", {k: round(v, 4) for k, v in T.items()})
with open(os.path.join(OUT, "test_predictions.tsv"), "w", encoding="utf-8") as fh:
    fh.write("ref\thyp\n"); [fh.write(f"{r}\t{h}\n") for r, h in zip(R, H)]
results = {"model": MODEL, "condition": COND, "family": FAMILY, "level": LEVEL, "seed": SEED, "config": CONFIG,
           "augment_max": A.MAX.get(FAMILY), "train_lines": len(train), "dev_lines": len(dev), "test_lines": len(test),
           "train_hours": sec / 3600, "dev_curve": curve, "final": {"test_greedy": T}}
json.dump(results, open(os.path.join(OUT, "results.json"), "w"), indent=1)

In [ ]:
from safetensors.torch import save_file
save_file({k: v.contiguous() for k, v in model.state_dict().items()}, os.path.join(OUT, "model.safetensors"))
json.dump({"chars": CHARS}, open(os.path.join(OUT, "charset.json"), "w"), ensure_ascii=False)
open(os.path.join(OUT, "README.md"), "w", encoding="utf-8").write(
    f"---\nlicense: mit\nlanguage:\n- he\ntags:\n- htr\n- ctc\n---\n\n# {OUT_REPO}\n\nAugmentation study (htr-metrics-transfer). "
    f"Model `{MODEL}`, augmentation `{COND}`, seed {SEED}. Trained on `cyttic/hhd-optionb-split` train minus the dev writers; "
    f"HHD Option-B test, greedy decoding, no LM: CER {T['cer']:.4f}, WER {T['wer']:.4f}. Details in results.json.\n")
if PUSH and not SMOKE:
    api = HfApi()
    for attempt in range(5):
        try:
            api.create_repo(OUT_REPO, private=False, exist_ok=True)
            api.upload_folder(repo_id=OUT_REPO, folder_path=OUT, commit_message=f"{MODEL} {COND} seed {SEED}"); break
        except Exception as e:
            print("upload failed, retrying:", repr(e)[:200]); time.sleep(30 * (attempt + 1))
    print("pushed -> https://huggingface.co/" + OUT_REPO)
else:
    print("not pushed; files in", OUT)